In [1]:
%load_ext dotenv
%dotenv

In [2]:
import os

SNOMEDCT_DIR = os.environ.get("SNOMEDCT")

In [3]:
from ehrax.example_schemes.snomed import SNOMEDCT

snomed = SNOMEDCT.from_gb_monolith_dir('snomed-ct', SNOMEDCT_DIR)

In [4]:
# snomed.save(f"{SNOMEDCT_DIR}/snomed.h5")
# snomed = SNOMEDCT.load(f"{SNOMEDCT_DIR}/snomed.h5")

In [4]:
df = snomed.as_dataframe()

In [5]:
df

,code,desc,inheres_in,property,units,characetrizes,direct_site,inherent_location
S-100000000,S-100000000,BITTER-3 (substance),NaN,NaN,NaN,NaN,NaN,NaN
S-10000006,S-10000006,Radiating chest pain (finding),NaN,NaN,NaN,NaN,NaN,NaN
S-1000001000000103,S-1000001000000103,Urine tryptophan:creatinine ratio (observable ...,NaN,NaN,NaN,NaN,NaN,NaN
S-1000001000004108,S-1000001000004108,Mismatch repair endonuclease PMS2 (substance),NaN,NaN,NaN,NaN,NaN,NaN
S-10000011000001109,S-10000011000001109,Nu-Hope belt left small 6460C 70cm-78cm length...,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
S-999981000000107,S-999981000000107,Urine valine:creatinine ratio (observable entity),NaN,NaN,NaN,NaN,NaN,NaN
S-9999811000001102,S-9999811000001102,Nu-Hope belt right large 6457 90cm-100cm lengt...,NaN,NaN,NaN,NaN,NaN,NaN
S-99999003,S-99999003,BISMUSAL SUSPENSION (substance),NaN,NaN,NaN,NaN,NaN,NaN
S-999991000000109,S-999991000000109,Urine tyrosine:creatinine ratio (observable en...,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
import importlib

from ehrax.example_schemes.snomed import SNOMEDCTGBMonolith
import ehrax

importlib.reload(ehrax.example_schemes.snomed)
refset = SNOMEDCTGBMonolith.process_refset(
    f'{SNOMEDCT_DIR}/Snapshot/Refset/Content/der2_Refset_SimpleMONOSnapshot_GB_20250730.txt')

In [7]:
member2set = refset.groupby('member')['refset'].apply(frozenset).to_dict()
set2member = refset.groupby('refset')['member'].apply(frozenset).to_dict()


In [8]:
kms_refset1 = set2member['S-1853551000000106']
kms_refset2 = set2member['S-999002881000000100']
kms_refset = kms_refset1 | kms_refset2

In [64]:
import pandas as pd

# quant = pd.read_csv('quantitative.csv')
# maybe_quant = pd.read_csv('maybe_quantitative.csv')
kms_snomed_events = pd.read_csv('kms_snomed_events.csv', dtype={'SNOMED_Concept_ID': str})
kms_snomed_events.loc[:, 'SNOMED_Concept_ID'] = 'S-' + kms_snomed_events.loc[:, 'SNOMED_Concept_ID']
kms_counts = kms_snomed_events.groupby('SNOMED_Concept_ID')['Count_SNOMED_ID'].sum()


In [100]:
secondary_obs = snomed.as_dataframe(sorted(kms_refset))
secondary_obs = secondary_obs.loc[:, ['code', 'desc', 'inheres_in', 'property', 'direct_site']]
secondary_obs = secondary_obs.assign(count_at_pc_kms=secondary_obs.code.map(kms_counts).fillna(0),
                                     top_10_pc_kms=secondary_obs.code.map(
                                         kms_counts > kms_counts.quantile(0.9)).fillna(False), )
secondary_obs = secondary_obs.set_index(['inheres_in', 'property', 'direct_site', 'code'], drop=True).sort_index()


/tmp/ipykernel_240082/576332354.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  kms_counts > kms_counts.quantile(0.9)).fillna(False), )


In [103]:
proximal_ci_keywords = list(sorted({
    "lipoprotein",
    "lymphocyte",
    "haemoglobin",
    "basophill",
    "corpuscular",
    "platelet",
    "thrombocyte",
    "reticulocyte",
    "reactive",
    "sphered",
    "monocyte",
    "neutrophill",
    "nucleated red",
    "phosphate",
    "pulse",
    "red blood cell",
    "erythrocyte",
    "rheumatoid",
    "shbg",
    "binding globulin",
    "systolic",
    "bilirubin",
    "total protein",
    "triglycerides",
    "urate",
    "urea",
    "vitamin d",
    "white blood cell",
    "leukocyte",
    "alanine",
    "albumin",
    "alkaline",
    "apolipoprotein",
    "aspartate",
    "aminotransferase",
    "basophill",
    "reactive",
    "calcium",
    "cholesterol",
    "creatinine",
    "cystatin",
    "diastolic",
    "eosinophill",
    "eosinophill",
    "glutamyltransferase",
    "glucose",
    "pressure",
    "glycated haemoglobin",
    "hba1c",
    "hdl",
    "haematocrit",
    "igf",
    "ldl",
    "low density"
}))
keyword_hits = [
    tuple(i for i, d in zip(secondary_obs.index, secondary_obs.desc) if k in d.lower())
    for k in proximal_ci_keywords
]
hit_indices = list(set().union(*keyword_hits))
secondary_obs = secondary_obs.assign(in_literature=secondary_obs.index.isin(hit_indices))
secondary_obs = secondary_obs.assign(aa_manual_exclude=False)

In [104]:
secondary_obs.to_excel("secondary_obs.xlsx")

In [98]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
